<a href="https://colab.research.google.com/github/doubletran/resnet14-cifar10-pytorch/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import torch
import torchvision
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms


## Datasets and Dataloaders

In [47]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 128

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

## Layers and Models

In [49]:
class ResBlock(nn.Module):
    def __init__(self,inplanes: int,planes: int, stride: int = 1, downsampling:bool =False,  padding:int = 1) -> None:
        super().__init__()
        self.conv1= nn.Conv2d(inplanes,planes,kernel_size=3,
                                        stride=stride,padding=padding,bias=False)

        self.bn1 = nn.BatchNorm2d(planes)
        #self.relu1 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(planes,planes,kernel_size=3,
                              stride=1,padding=padding,bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.relu2 = nn.ReLU(inplace=True)
        self.shortcut = nn.Identity()

        if stride != 1 or inplanes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(inplanes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = F.relu(out)
        #out = self.relu1(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out +=self.shortcut(x)
        out = self.relu2(out)

        return out

In [62]:
class MyResNet(nn.Module):

  def __init__(self,inplanes: int, classes: int,) -> None:
    num_filters=16
    super().__init__()
    self.conv1 = nn.Conv2d(inplanes,num_filters,kernel_size=3, padding=1, bias=True)
    inplanes=32
    self.layer1 = self.make_layer(16, 16, 1)
    self.layer2 = self.make_layer(16, 32, 2)
    self.layer3 = self.make_layer(32, 64, 2)
    self.globalpool = nn.AvgPool2d((8, 8))
    self.fc = nn.Linear(64, classes)
  def make_layer(self, inplanes, planes, stride):
    n = 2
    blocks = [ResBlock(inplanes, planes, stride), ResBlock(planes, planes, 1)]

    return nn.Sequential(*blocks)


  def forward(self, x: Tensor) -> Tensor:
    out = self.conv1(x)
    out = F.relu(out)
    out = self.layer1(out)
    out = self.layer2(out)
    out = self.layer3(out)
    #out = self.conv1x1(out)

    out = self.globalpool(out)
    out = torch.flatten(out, 1)
    out = self.fc(out)
    return out.squeeze()

In [63]:
model = MyResNet(3, 10)
print(model)

MyResNet(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (layer1): Sequential(
    (0): ResBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu2): ReLU(inplace=True)
      (shortcut): Identity()
    )
    (1): ResBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu2): ReLU(inplace=True)
     

In [56]:
import matplotlib.pyplot as plt
import numpy as np

def visualizeBatch(x_batch,pred, num=10):
  f, axs = plt.subplots(num, 2, figsize=(14,6*num))
  print(pred.shape)
  print(x_batch.shape)
  for i in range(0,num):
    img = x_batch[i,:,:].squeeze()
    img = img / 2 + 0.5
    axs[i,0].imshow(np.transpose(img, (1, 2, 0)), cmap='gray')
    axs[i,1].bar(list(range(10)),pred[i,:], label=list(classes))

  return f


## Training

In [65]:
import wandb

wandb.login()


device = torch.device('cuda')

model.to(device)


config = {'epochs': 20, 'lr': 3e-3}
iter = 0
with wandb.init(config = config) as run:

    wandb.define_metric("train/iter_loss", step_metric="global_step")
    wandb.define_metric("val/epoch_loss", step_metric="epoch")
    wandb.define_metric("val/epoch_accuracy", step_metric="epoch")
    wandb.define_metric("val/epoch_err", step_metric="epoch")

    optimizer = torch.optim.SGD(model.parameters(), lr=run.config['lr'], weight_decay=0.0)
    criterion = torch.nn.CrossEntropyLoss()
    for i in range(run.config['epochs']):
        model.train()
        print("Epoch {}".format(i))
        for j,input in enumerate(trainloader,0):
            iter+=1



            x = input[0].to(device)
            y = input[1].to(device)

            if iter == 0:
                image = wandb.Image(x[0,:,:,:])
                run.log({"example": image})

            out = model(x)
            loss = criterion(out,y)
            model.conv1.weight.requires_grad = False

            model.zero_grad()
            loss.backward()

            optimizer.step()

            _, predicted = torch.max(out.data, 1)
            correct = (predicted == y).float().mean().item()


            run.log({"train/train_loss": loss.item(), "train/train_accuracy": correct}, step = iter)



        model.eval()
        running_loss = 0
        running_acc = 0
        running_err = 0
        for j,input in enumerate(testloader,0):

             x = input[0].to(device)
             y = input[1].to(device)

             out = model(x)
             loss = criterion(out,y)

             _, predicted = torch.max(out.data, 1)
             correct = (predicted == y).sum().item()

             running_loss += loss.item()
             running_acc += correct
             error = (predicted != y).sum().item()
             running_err += error
             if j == 0:
                  f = visualizeBatch(x.detach().cpu(),F.softmax(out,dim=1).detach().cpu())
                  run.log({"batch0": f})
                  plt.close(f)


        run.log({"epoch": i, "val/epoch_loss": running_loss / len(testloader), "val/epoch_accuracy": running_acc / len(testloader), "val/epoch_error": running_err/len(testloader)}, step = iter + 1)


Epoch 0
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 1
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 2
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 3
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 4
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 5
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 6
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 7
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 8
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 9
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 10
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 11
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 12
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 13
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 14
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 15
torch.Size([128, 10])
torch.Size([128, 3, 32, 32])
Epoch 16
torch.Size([128, 10])
torch.Size([128, 3,

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/train_accuracy,▄▃▃▄▄▄▄▄▅▄▁▆▆▅▄▅▅▄▆▂▆▅▇▅▆▄▅▇▆▇▆▅▅▇▆▇▇▇▇█
train/train_loss,█▆▇▇█▄▅▇█▃▆▄▂▅▂▆▅▄▆▃▁▄▄▂▁▅▃▃▄▄▃▂▄█▄▁▄▅▂▁
val/epoch_accuracy,▁▃▅▄▁▄▇▆▆▅▂▇▂▇▆▇█▆█▇
val/epoch_error,█▆▄▅█▅▂▃▃▄▇▂▇▂▃▂▁▃▁▂
val/epoch_loss,▇▆▄▅█▄▂▃▃▄▆▂▇▂▃▂▁▃▁▂
epoch,19
train/train_accuracy,0.8625
train/train_loss,0.55982
val/epoch_accuracy,86.78481
val/epoch_error,39.79747
